# Pipeline de préparation des données ScalBnB

Génère `data/master_final.csv` à partir des données brutes Inside Airbnb
(`listings.csv`, `calendar.csv`), placées dans `data/raw/`.

Les fichiers bruts ne sont pas versionnés (trop volumineux) — à télécharger sur
[Inside Airbnb](http://insideairbnb.com/get-the-data/) (Bordeaux Métropole).

## 1. Taux d'occupation annuel (`calendar.csv`)

In [1]:
import pandas as pd

calendar = pd.read_csv("../data/raw/calendar.csv")
calendar["date"] = pd.to_datetime(calendar["date"])

# 'available' vaut 'f' quand le logement est occupé ce jour-là : on inverse
# pour obtenir un indicateur d'occupation direct (1 = occupé, 0 = libre).
calendar["available_norm"] = calendar["available"].map({"f": 1, "t": 0})

occupation_annuelle = (
    calendar.groupby("listing_id")["available_norm"]
    .mean()
    .reset_index()
    .rename(columns={"available_norm": "taux_occupation"})
)
occupation_annuelle.head()

,listing_id,taux_occupation
0,222887,0.189041
1,317273,0.180822
2,317658,0.205479
3,333031,0.052055
4,365993,0.257534


## 2. Nettoyage des annonces (`listings.csv`)

In [2]:
listings = pd.read_csv("../data/raw/listings.csv")

# On se concentre sur les locations courte durée, le public cible de l'app
listings = listings[listings["minimum_nights"] <= 30].copy()

colonnes_utiles = [
    "id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
    "price", "minimum_nights", "maximum_nights", "amenities",
    "review_scores_rating", "estimated_occupancy_l365d",
]
listings = listings[colonnes_utiles].copy()

listings["price"] = (
    listings["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)
listings.head()

,id,neighbourhood_cleansed,room_type,accommodates,bedrooms,price,minimum_nights,maximum_nights,amenities,review_scores_rating,estimated_occupancy_l365d
0,222887,Bordeaux Sud,Entire home/apt,4,2.0,241.0,3,90,"[""Host greets you"", ""Refrigerator"", ""Fast wifi...",4.83,168
1,317273,Chartrons - Grand Parc - Jardin Public,Entire home/apt,3,1.0,214.0,3,90,"[""Host greets you"", ""Refrigerator"", ""Courtyard...",4.91,114
2,317658,Centre ville (Bordeaux),Entire home/apt,6,2.0,246.0,3,90,"[""Host greets you"", ""Refrigerator"", ""Toaster"",...",4.87,84
3,333031,Centre ville (Bordeaux),Entire home/apt,2,0.0,104.0,1,1125,"[""Refrigerator"", ""Hair dryer"", ""Lockbox"", ""TV ...",4.93,255
4,365993,Bgles,Entire home/apt,6,2.0,77.0,5,30,"[""Host greets you"", ""Room-darkening shades"", ""...",4.90,100


## 3. Extraction des équipements clés

On se limite à 5 équipements à fort impact plutôt qu'à l'ensemble des
équipements possibles : ça donne un profil de comparaison lisible, et ce
sont ceux que l'application compare ensuite au segment de référence.

In [3]:
listings["Wifi"] = listings["amenities"].str.lower().str.contains("wifi").astype(int)
listings["kitchen"] = listings["amenities"].str.lower().str.contains("kitchen").astype(int)

# (?<!dish) évite de confondre lave-linge et lave-vaisselle
listings["has_washer"] = (
    listings["amenities"].str.lower().str.contains(r"(?<!dish)washer", regex=True, na=False)
).astype(int)

listings["air conditioning"] = (
    listings["amenities"].str.lower().str.contains("air conditioning").astype(int)
)


def a_un_parking_gratuit(amenities_str: str) -> int:
    """Un logement a un parking gratuit sur place si un équipement mentionne
    à la fois 'free', 'on premises' et un mot-clé parking/garage/driveway."""
    import ast
    for equip in ast.literal_eval(amenities_str):
        e = equip.lower()
        est_parking = "parking" in e or "garage" in e or "driveway" in e
        if est_parking and "free" in e and "on premises" in e:
            return 1
    return 0


listings["parking_gratuit"] = listings["amenities"].apply(a_un_parking_gratuit)
listings[["Wifi", "kitchen", "has_washer", "air conditioning", "parking_gratuit"]].mean()

Wifi                0.931671
kitchen             0.907189
has_washer          0.716635
air conditioning    0.257148
parking_gratuit     0.510825
dtype: float64

## 4. Segmentation : quartier + type de logement + gamme de prix

In [4]:
master = listings.merge(
    occupation_annuelle, left_on="id", right_on="listing_id", how="left"
).drop(columns=["listing_id"])

GROUP_COLS = ["room_type", "neighbourhood_cleansed"]


def tercile_prix(prix: pd.Series) -> pd.Series:
    try:
        return pd.qcut(prix, q=3, labels=["bas", "moyen", "haut"], duplicates="drop")
    except ValueError:
        return pd.Series("unique", index=prix.index)


tranche_prix = master.groupby(GROUP_COLS)["price"].transform(tercile_prix)
master["tranche_prix"] = tranche_prix
GROUP_COLS_COMPLET = GROUP_COLS + ["tranche_prix"]
master["tranche_prix"].value_counts()

tranche_prix
bas       2640
haut      2463
moyen     2424
unique      48
Name: count, dtype: int64

## 5. Classement Top / Standard par segment

Un logement est "Top" s'il fait partie du quart le plus occupé (percentile 75)
de son segment. En dessous de 10 logements comparables, le segment est jugé
trop petit pour un diagnostic fiable (`classement = -1`).

In [5]:
import numpy as np

master["seuil"] = master.groupby(GROUP_COLS_COMPLET)["estimated_occupancy_l365d"].transform(
    lambda occ: occ.quantile(0.75)
)
master["classement"] = np.where(master["estimated_occupancy_l365d"] >= master["seuil"], 1, 0)

master["taille_groupe"] = master.groupby(GROUP_COLS_COMPLET)["id"].transform("count")
master["classement"] = np.where(master["taille_groupe"] <= 10, -1, master["classement"])

master["classement"].value_counts()

classement
 0    8177
 1    1784
-1     986
Name: count, dtype: int64

## 6. Profil de référence des Tops et fusion

On calcule, pour chaque segment, la proportion de Tops qui possèdent chaque
équipement et leur nombre médian de nuits minimum. La fusion avec `master`
crée automatiquement les suffixes `_x` (valeur du logement) et `_y` (valeur de
référence du segment) — c'est cette paire que l'application compare ensuite
pour générer ses recommandations.

In [6]:
EQUIP_COLS = ["Wifi", "kitchen", "has_washer", "air conditioning", "parking_gratuit"]

profil_tops = (
    master[master["classement"] == 1]
    .groupby(GROUP_COLS_COMPLET)[EQUIP_COLS + ["minimum_nights"]]
    .agg({**{col: "mean" for col in EQUIP_COLS}, "minimum_nights": "median"})
    .reset_index()
)

master = master.merge(profil_tops, on=GROUP_COLS_COMPLET, how="left", suffixes=("_x", "_y"))
master = master.drop(columns=["tranche_prix"])  # utile seulement pour le regroupement ci-dessus
master.filter(like="Wifi").head()

,Wifi_x,Wifi_y
0,1,1.00000
1,1,1.00000
2,1,1.00000
3,1,0.97619
4,1,1.00000


## 7. Recommandations par équipement

Un équipement est recommandé si le logement ne l'a pas (`_x == 0`) alors
qu'au moins la moitié des Tops du segment l'ont (`_y >= 0.5`). Même logique
pour les nuits minimum, avec une marge de 2 nuits pour éviter de recommander
des ajustements trop marginaux.

In [7]:
SEUIL_EQUIP = 0.5
MARGE_NUITS = 2

master["recor_wifi"] = np.where((master["Wifi_x"] == 0) & (master["Wifi_y"] >= SEUIL_EQUIP), 1, np.nan)
master["reco_kitchen"] = np.where((master["kitchen_x"] == 0) & (master["kitchen_y"] >= SEUIL_EQUIP), 1, np.nan)
master["reco_washer"] = np.where((master["has_washer_x"] == 0) & (master["has_washer_y"] >= SEUIL_EQUIP), 1, np.nan)
master["reco_ac"] = np.where(
    (master["air conditioning_x"] == 0) & (master["air conditioning_y"] >= SEUIL_EQUIP), 1, np.nan
)
master["reco_parking"] = np.where(
    (master["parking_gratuit_x"] == 0) & (master["parking_gratuit_y"] >= SEUIL_EQUIP), 1, np.nan
)
master["reco_nuit"] = np.where(
    master["minimum_nights_x"] >= master["minimum_nights_y"] + MARGE_NUITS, 1, np.nan
)

RECO_COLS = ["recor_wifi", "reco_kitchen", "reco_washer", "reco_ac", "reco_parking", "reco_nuit"]
master[RECO_COLS].notna().sum()

recor_wifi       427
reco_kitchen     408
reco_washer      960
reco_ac          639
reco_parking     760
reco_nuit       1381
dtype: int64

## 8. Texte de recommandation lisible

Pour l'affichage dans l'app (onglet Diagnostic), on combine les
recommandations actives d'un logement en un texte unique.

In [8]:
LIBELLES_RECO = {
    "recor_wifi": "Ajouter le Wifi",
    "reco_kitchen": "Ajouter une cuisine équipée",
    "reco_washer": "Installer un lave-linge",
    "reco_ac": "Installer la climatisation",
    "reco_parking": "Proposer un parking gratuit",
    "reco_nuit": "Réduire le minimum de nuits",
}


def construire_texte_recommandation(row: pd.Series) -> str:
    if row["classement"] == -1:
        return "Pas assez de logements comparables dans votre segment pour un diagnostic fiable."
    if row["classement"] == 1:
        return "Votre logement fait déjà partie des meilleurs de votre segment. Continuez ainsi !"

    actions = [libelle for col, libelle in LIBELLES_RECO.items() if pd.notna(row[col])]
    if not actions:
        return "Votre logement correspond déjà au profil des meilleurs de votre segment."

    return "\n".join(f"{i}. {action}" for i, action in enumerate(actions, 1))


master["recommandation_finale"] = master.apply(construire_texte_recommandation, axis=1)
master["recommandation_finale"].value_counts().head(10)

recommandation_finale
Votre logement correspond déjà au profil des meilleurs de votre segment.             5521
Votre logement fait déjà partie des meilleurs de votre segment. Continuez ainsi !    1784
Pas assez de logements comparables dans votre segment pour un diagnostic fiable.      986
1. Réduire le minimum de nuits                                                        748
1. Installer un lave-linge                                                            356
1. Proposer un parking gratuit                                                        273
1. Installer la climatisation                                                         237
1. Ajouter le Wifi                                                                    167
1. Ajouter une cuisine équipée                                                        127
1. Installer la climatisation\n2. Réduire le minimum de nuits                         107
Name: count, dtype: int64

## 9. Export

In [9]:
master.to_csv("../data/master_final.csv", index=False)
print(f"master_final.csv exporté : {master.shape[0]} lignes, {master.shape[1]} colonnes")

master_final.csv exporté : 10947 lignes, 33 colonnes
